In [1]:
import pandas as pd
import os 
from utils.dataframe_tools import truncate_to_block_by_schema 
from utils.pruning_and_merge import merge_field_blocks, merge_field_blocks_tree_similarity

In [2]:
import torch

print(torch.__version__)

2.5.1


In [2]:
ori_csv_path = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'completeness', 'dataset_29_completed_label.csv')
output_dir_ori = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'reborn_blocks_ori')
output_dir_merge = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'reborn_blocks_merge')
output_dir_merge_ps = os.path.join('..', 'TrafficData', 'dataset_29_d1_csv_merged', 'reborn_blocks_merge_ps')

In [4]:
truncate_to_block_by_schema(ori_csv_path, output_dir_ori)

Data loading completed, 1304943 records here. 
Fingerprints has been built. 
There are 242 Field Block in total. 


Saving blocks: 100%|██████████| 242/242 [00:18<00:00, 12.95it/s] 


Truncation succeeded! 


In [5]:
merge_field_blocks(output_dir_ori, output_dir_merge, jaccard_threshold=0.8)

开始执行Field Block合并与剪枝流程...

[步骤 1/5] 正在扫描并分析所有 242 个原始Block...


Scanning blocks: 100%|██████████| 242/242 [00:05<00:00, 41.43it/s] 



[步骤 2/5] 识别完成:
 - 20 个核心专家 (样本数 >= 1000)
 - 222 个待合并的小型专家

[步骤 3/5] 正在为小型专家寻找最佳合并目标...


Merging small blocks: 100%|██████████| 222/222 [00:00<00:00, 321.74it/s] 



[步骤 4/5] 正在处理 156 个“孤儿”Block...

[步骤 5/5] 正在保存 21 个最终的Block...


Saving final blocks: 100%|██████████| 21/21 [00:07<00:00,  2.73it/s]



合并与剪枝成功！最终生成了 21 个专家Block。
合并日志已保存到: ..\TrafficData\dataset_29_d1_csv_merged\reborn_blocks_merge\merge_log.json


In [3]:
merge_field_blocks_tree_similarity(output_dir_ori, output_dir_merge_ps, similarity_threshold=0.8)

开始执行基于【结构化树相似度】的Field Block合并与剪枝流程...

[步骤 1/5] 正在扫描并分析所有原始Block...


Scanning blocks: 100%|██████████| 242/242 [00:03<00:00, 75.25it/s]



[步骤 2/5] 识别完成:
 - 20 个核心专家 (样本数 >= 1000)
 - 222 个待合并的小型专家

[步骤 3/5] 正在为小型专家寻找最佳合并目标...


Merging small blocks: 100%|██████████| 222/222 [00:01<00:00, 164.12it/s]



[步骤 4/5] 正在处理 161 个“孤儿”Block...

[步骤 5/5] 正在保存 21 个最终的Block...


Saving final blocks: 100%|██████████| 21/21 [00:07<00:00,  2.71it/s]



合并与剪枝成功！最终生成了 21 个专家Block。
合并日志已保存到: ..\TrafficData\dataset_29_d1_csv_merged\reborn_blocks_merge_ps\merge_log.json


In [3]:
from utils.dataframe_tools import protocol_tree, add_root_layer 
test_block_1 = os.path.join(output_dir_merge, '24.csv') 
test_block_2 = os.path.join(output_dir_merge, '35.csv')
test_default_block = os.path.join(output_dir_merge, 'default_expert.csv')

In [4]:
df_block_1 = pd.read_csv(test_block_1) 
df_block_2 = pd.read_csv(test_block_2)

ptree_1 = protocol_tree(df_block_1) 
ptree_2 = protocol_tree(df_block_2)

C:\Users\PC\AppData\Local\Temp\ipykernel_15660\3799198723.py:1: DtypeWarning: Columns (4,20,33,34,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_block_1 = pd.read_csv(test_block_1)
C:\Users\PC\AppData\Local\Temp\ipykernel_15660\3799198723.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_block_2 = pd.read_csv(test_block_2)


In [5]:
df_default_block = pd.read_csv(test_default_block) 
ptree_default = protocol_tree(df_default_block)

In [7]:
from utils.pruning_and_merge import calculate_tree_similarity

p1_vs_p2, node_sim1, edge_sim1 = calculate_tree_similarity(ptree_1, ptree_2)
p1_vs_pd, node_sim2, edge_sim2 = calculate_tree_similarity(ptree_1, ptree_default)
print("="*50)
print("结构化树相似度计算结果:")
print("="*50)
print(f"pt1 vs pt2:")
print(f"  - 节点相似度: {node_sim1:.4f}")
print(f"  - 结构 (边) 相似度: {edge_sim1:.4f}")
print(f"  - ===> 最终相似度: {p1_vs_p2:.4f} <===")
print("="*50)
print(f"pt1 vs ptd:")
print(f"  - 节点相似度: {node_sim2:.4f}")
print(f"  - 结构 (边) 相似度: {edge_sim2:.4f}")
print(f"  - ===> 最终相似度: {p1_vs_pd:.4f} <===")


结构化树相似度计算结果:
pt1 vs pt2:
  - 节点相似度: 0.8906
  - 结构 (边) 相似度: 0.8814
  - ===> 最终相似度: 0.8860 <===
pt1 vs ptd:
  - 节点相似度: 0.2365
  - 结构 (边) 相似度: 0.2203
  - ===> 最终相似度: 0.2284 <===


In [5]:
print(ptree_1)

{'eth': ['eth.dst', 'eth.src', 'eth.type'], 'ip': ['ip.version', 'ip.hdr_len', 'ip.dsfield', 'ip.len', 'ip.id', 'ip.flags', 'ip.frag_offset', 'ip.ttl', 'ip.proto', 'ip.checksum', 'ip.src', 'ip.dst'], 'tcp': ['tcp.srcport', 'tcp.dstport', 'tcp.stream', 'tcp.len', 'tcp.seq', 'tcp.seq_raw', 'tcp.ack', 'tcp.ack_raw', 'tcp.hdr_len', 'tcp.flags', 'tcp.window_size_value', 'tcp.window_size', 'tcp.window_size_scalefactor', 'tcp.checksum', 'tcp.urgent_pointer'], 'tls': [], 'statistics': ['index', 'label'], 'eth.dst': ['eth.dst.lg', 'eth.dst.ig'], 'eth.src': ['eth.src.lg', 'eth.src.ig'], 'ip.dsfield': ['ip.dsfield.dscp', 'ip.dsfield.ecn'], 'ip.flags': ['ip.flags.rb', 'ip.flags.df', 'ip.flags.mf'], 'tcp.flags': ['tcp.flags.res', 'tcp.flags.ae', 'tcp.flags.cwr', 'tcp.flags.ece', 'tcp.flags.urg', 'tcp.flags.ack', 'tcp.flags.push', 'tcp.flags.reset', 'tcp.flags.syn', 'tcp.flags.fin', 'tcp.flags.str']}


In [11]:
print(ptree_2)

{'eth': ['eth.dst', 'eth.src', 'eth.type'], 'ip': ['ip.version', 'ip.hdr_len', 'ip.dsfield', 'ip.len', 'ip.id', 'ip.flags', 'ip.frag_offset', 'ip.ttl', 'ip.proto', 'ip.checksum', 'ip.src', 'ip.dst'], 'tcp': ['tcp.srcport', 'tcp.dstport', 'tcp.stream', 'tcp.len', 'tcp.seq', 'tcp.seq_raw', 'tcp.ack', 'tcp.ack_raw', 'tcp.hdr_len', 'tcp.flags', 'tcp.window_size_value', 'tcp.window_size', 'tcp.window_size_scalefactor', 'tcp.checksum', 'tcp.urgent_pointer', 'tcp.option_kind', 'tcp.option_len', 'tcp.options'], 'tls': [], 'statistics': ['index', 'label'], 'eth.dst': ['eth.dst.lg', 'eth.dst.ig'], 'eth.src': ['eth.src.lg', 'eth.src.ig'], 'ip.dsfield': ['ip.dsfield.dscp', 'ip.dsfield.ecn'], 'ip.flags': ['ip.flags.rb', 'ip.flags.df', 'ip.flags.mf'], 'tcp.flags': ['tcp.flags.res', 'tcp.flags.ae', 'tcp.flags.cwr', 'tcp.flags.ece', 'tcp.flags.urg', 'tcp.flags.ack', 'tcp.flags.push', 'tcp.flags.reset', 'tcp.flags.syn', 'tcp.flags.fin', 'tcp.flags.str'], 'tcp.options': ['tcp.options.nop', 'tcp.options.

In [10]:
print(ptree_default)

{'eth': ['eth.dst', 'eth.src', 'eth.type'], 'ip': ['ip.version', 'ip.hdr_len', 'ip.dsfield', 'ip.len', 'ip.id', 'ip.flags', 'ip.frag_offset', 'ip.ttl', 'ip.proto', 'ip.checksum', 'ip.src', 'ip.dst'], 'tcp': ['tcp.srcport', 'tcp.dstport', 'tcp.stream', 'tcp.len', 'tcp.seq', 'tcp.seq_raw', 'tcp.ack', 'tcp.ack_raw', 'tcp.hdr_len', 'tcp.flags', 'tcp.window_size_value', 'tcp.window_size', 'tcp.window_size_scalefactor', 'tcp.checksum', 'tcp.urgent_pointer', 'tcp.options', 'tcp.option_kind', 'tcp.option_len'], 'tls': ['tls.record', 'tls.handshake', 'tls.x509af', 'tls.x509if', 'tls.x509sat', 'tls.pkcs1', 'tls.x509ce', 'tls.ber', 'tls.pkix1implicit', 'tls.ocsp', 'tls.cms', 'tls.pkix1explicit', 'tls.logotypecertextn', 'tls.sct', 'tls.alert_message', 'tls.ns_cert_exts', 'tls.ns'], 'statistics': ['index', 'label'], 'eth.dst': ['eth.dst.lg', 'eth.dst.ig'], 'eth.src': ['eth.src.lg', 'eth.src.ig'], 'ip.dsfield': ['ip.dsfield.dscp', 'ip.dsfield.ecn'], 'ip.flags': ['ip.flags.rb', 'ip.flags.df', 'ip.fla

In [ ]:
# available_fields = set(dataframe.columns)
# configured_fields = set(self.config.keys())
# self.real_nodes = sorted(list(available_fields.intersection(configured_fields)))
# self.ptree = protocol_tree(self.real_nodes)